In [0]:
# Databricks notebook source
# ==============================================================================
# TrackPulse — Publish to Azure SQL Data Warehouse
# Notebook: 04_publish_azure_sql_prod
# Layer: Lakehouse (Silver & Gold) -> Serving Layer (Azure SQL dw Schema)
# Strictly adheres to MD Sections 6, 7 & 10
# ==============================================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------------------------
# 1. Connection Configurations & Parameters
# ------------------------------------------------------------------------------
sql_password = "samsung#123"
jdbc_server = "sqlsic.database.windows.net"
jdbc_port = 1433
jdbc_database = "sqldb-trackpulse-gold"
jdbc_user = "sqladmin"

jdbc_url = f"jdbc:sqlserver://{jdbc_server}:{jdbc_port};database={jdbc_database};encrypt=true;trustServerCertificate=false;loginTimeout=60;"

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

def write_to_sql(df, table_name, mode="append"):
    """Writes a Spark DataFrame directly to an Azure SQL table via JDBC."""
    count = df.count()
    if count == 0:
        print(f"No new records to write to {table_name}. Skipping...")
        return
    print(f"Writing {count} rows to {table_name} (mode: {mode})...")
    df.write \
        .format("jdbc") \
        .mode(mode) \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("user", jdbc_user) \
        .option("password", sql_password) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .option("batchsize", "10000") \
        .save()
    print(f"Successfully published {table_name}")

def read_from_sql(table_name):
    """Reads a table from Azure SQL into a Spark DataFrame."""
    return spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("user", jdbc_user) \
        .option("password", sql_password) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()

def is_table_loaded(table_name):
    """Checks whether the target SQL table already contains records."""
    try:
        sample = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", f"(SELECT TOP 1 * FROM {table_name}) AS chk") \
            .option("user", jdbc_user) \
            .option("password", sql_password) \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .load()
        return sample.count() > 0
    except:
        return False

def get_field(df, candidates, target_type):
    """Safely extracts the first matching column or falls back to a cast NULL literal."""
    for c in candidates:
        if c in df.columns:
            return F.col(c).cast(target_type)
    return F.lit(None).cast(target_type)

# ------------------------------------------------------------------------------
# 2. Step 1: Check Dimension Tables (MD Section 6.1)
# ------------------------------------------------------------------------------
if is_table_loaded("dw.dim_vehicle"):
    print("dw.dim_vehicle already populated. Skipping...")
if is_table_loaded("dw.dim_driver"):
    print("dw.dim_driver already populated. Skipping...")
if is_table_loaded("dw.dim_route"):
    print("dw.dim_route already populated. Skipping...")
if is_table_loaded("dw.dim_shipment"):
    print("dw.dim_shipment already populated. Skipping...")

# ------------------------------------------------------------------------------
# 3. Step 2: Read Resolved Surrogate Keys from SQL (MD Section 10.5)
# ------------------------------------------------------------------------------
dim_vehicle_lookup = read_from_sql("dw.dim_vehicle").select("vehicle_id", "vehicle_key")
dim_driver_lookup = read_from_sql("dw.dim_driver").select("driver_id", "driver_key")
dim_route_lookup = read_from_sql("dw.dim_route").select("route_id", "route_key")
dim_shipment_lookup = read_from_sql("dw.dim_shipment").select(
    "shipment_id", "shipment_key", "required_temp_min_c", "required_temp_max_c"
)
dim_date_lookup = read_from_sql("dw.dim_date").select("full_date", "date_key")

default_route_key = dim_route_lookup.select("route_key").first()[0]

# ------------------------------------------------------------------------------
# 4. Step 3: Load Fact Tables (MD Section 6.2)
# ------------------------------------------------------------------------------

# 4.1 Fact 1: dw.fct_trip
if is_table_loaded("dw.fct_trip"):
    print("dw.fct_trip already populated. Skipping...")
else:
    gold_trip_df = spark.read.format("delta").load(storage_path + "gold/trip_summary/")
    fct_trip_df = gold_trip_df \
        .join(dim_vehicle_lookup, "vehicle_id", "inner") \
        .join(dim_driver_lookup, "driver_id", "inner") \
        .join(dim_route_lookup, "route_id", "inner") \
        .join(dim_shipment_lookup.select("shipment_id", "shipment_key"), "shipment_id", "inner") \
        .withColumn("departure_date", F.to_date(get_field(gold_trip_df, ["actual_departure_time", "start_time"], "timestamp"))) \
        .withColumn("arrival_date", F.to_date(get_field(gold_trip_df, ["actual_arrival_time", "end_time"], "timestamp"))) \
        .join(dim_date_lookup.withColumnRenamed("date_key", "departure_date_key"), 
              F.col("departure_date") == F.col("full_date"), "left").drop("full_date", "departure_date") \
        .join(dim_date_lookup.withColumnRenamed("date_key", "arrival_date_key"), 
              F.col("arrival_date") == F.col("full_date"), "left").drop("full_date", "arrival_date") \
        .select(
            get_field(gold_trip_df, ["trip_id"], "string").alias("trip_id"),
            F.col("vehicle_key").cast("int"),
            F.col("driver_key").cast("int"),
            F.col("route_key").cast("int"),
            F.col("shipment_key").cast("int"),
            F.col("departure_date_key").cast("int"),
            F.col("arrival_date_key").cast("int"),
            get_field(gold_trip_df, ["planned_departure_time"], "timestamp").alias("planned_departure_time"),
            get_field(gold_trip_df, ["planned_arrival_time"], "timestamp").alias("planned_arrival_time"),
            get_field(gold_trip_df, ["actual_departure_time", "start_time"], "timestamp").alias("actual_departure_time"),
            get_field(gold_trip_df, ["actual_arrival_time", "end_time"], "timestamp").alias("actual_arrival_time"),
            get_field(gold_trip_df, ["distance_km", "total_distance_km"], "decimal(10,2)").alias("distance_km"),
            get_field(gold_trip_df, ["fuel_used_liters", "total_fuel_liters"], "decimal(10,2)").alias("fuel_used_liters"),
            get_field(gold_trip_df, ["duration_minutes", "total_duration_minutes"], "decimal(10,2)").alias("duration_minutes"),
            get_field(gold_trip_df, ["delay_minutes"], "decimal(10,2)").alias("delay_minutes"),
            get_field(gold_trip_df, ["on_time_flag"], "boolean").alias("on_time_flag"),
            get_field(gold_trip_df, ["trip_status"], "string").alias("trip_status")
        ).dropDuplicates(["trip_id"])
    write_to_sql(fct_trip_df, "dw.fct_trip", mode="append")

# 4.2 Fact 2: dw.fct_cold_chain (Incremental by Episode ID)
gold_cold_df = spark.read.format("delta").load(storage_path + "gold/cold_chain_daily_summary/")
gold_trip_df = spark.read.format("delta").load(storage_path + "gold/trip_summary/")

trip_route_map = gold_trip_df.select("shipment_id", "route_id").dropDuplicates(["shipment_id"])

fct_cold_df = gold_cold_df \
    .join(dim_shipment_lookup, "shipment_id", "inner") \
    .join(dim_vehicle_lookup, "vehicle_id", "inner") \
    .join(trip_route_map, "shipment_id", "left") \
    .join(dim_route_lookup, "route_id", "left") \
    .withColumn("date_key", F.date_format(F.col("summary_date"), "yyyyMMdd").cast("int")) \
    .withColumn("resolved_route_key", F.coalesce(F.col("route_key"), F.lit(default_route_key)).cast("int")) \
    .withColumn("breach_count", F.coalesce(F.col("readings_above_required_max"), F.lit(0)) + 
                                F.coalesce(F.col("readings_below_required_min"), F.lit(0))) \
    .select(
        F.concat(F.lit("EP_"), F.col("shipment_id"), F.lit("_"), F.date_format(F.col("summary_date"), "yyyyMMdd")).alias("breach_episode_id"),
        F.col("shipment_key").cast("int"),
        F.col("vehicle_key").cast("int"),
        F.col("resolved_route_key").alias("route_key"),
        F.col("date_key"),
        F.to_timestamp(F.col("summary_date")).alias("start_time"),
        F.to_timestamp(F.col("summary_date")).alias("end_time"),
        F.col("breach_count").cast("decimal(10,2)").alias("duration_minutes"),
        F.col("min_cargo_temp_c").cast("decimal(5,2)").alias("min_observed_temp_c"),
        F.col("max_cargo_temp_c").cast("decimal(5,2)").alias("max_observed_temp_c"),
        F.col("required_temp_min_c").cast("decimal(5,2)"),
        F.col("required_temp_max_c").cast("decimal(5,2)"),
        F.when(F.col("breach_count") > 15, "CRITICAL")
         .when(F.col("breach_count") > 5, "HIGH")
         .otherwise("MEDIUM").alias("severity"),
        F.lit("CLOSED").alias("episode_status")
    ).dropDuplicates(["breach_episode_id"])

# Filter out episodes already published to SQL
if is_table_loaded("dw.fct_cold_chain"):
    existing_episodes = read_from_sql("(SELECT DISTINCT breach_episode_id FROM dw.fct_cold_chain) AS existing_ep")
    fct_cold_df = fct_cold_df.join(existing_episodes, "breach_episode_id", "left_anti")

write_to_sql(fct_cold_df, "dw.fct_cold_chain", mode="append")

# 4.3 Fact 3: dw.fct_vehicle_telemetry (Incremental Publishing with Watermarking)
# Query the watermark (latest event_time in SQL table)
watermark_query = "(SELECT ISNULL(MAX(event_time), '1970-01-01 00:00:00') AS max_watermark FROM dw.fct_vehicle_telemetry) AS wm"
sql_watermark_ts = read_from_sql(watermark_query).first()["max_watermark"]
print(f"Current Watermark in dw.fct_vehicle_telemetry: {sql_watermark_ts}")

# Read telemetry stream buffer and filter by watermark
silver_telemetry_stream_path = storage_path + "silver/telemetry_stream/"
silver_telemetry_df = spark.read.format("delta").load(silver_telemetry_stream_path)

event_time_col = get_field(silver_telemetry_df, ["event_timestamp", "event_time"], "timestamp")

# Filter new telemetry events strictly after the watermark
incremental_telemetry_df = silver_telemetry_df.filter(event_time_col > F.lit(sql_watermark_ts))

fct_telemetry_df = incremental_telemetry_df \
    .join(dim_vehicle_lookup, "vehicle_id", "inner") \
    .join(dim_driver_lookup, "driver_id", "left") \
    .join(dim_route_lookup, "route_id", "left") \
    .join(dim_shipment_lookup.select("shipment_id", "shipment_key"), "shipment_id", "left") \
    .withColumn("date_key", F.date_format(event_time_col, "yyyyMMdd").cast("int")) \
    .select(
        get_field(incremental_telemetry_df, ["event_id"], "string").alias("event_id"),
        event_time_col.alias("event_time"),
        F.col("vehicle_key").cast("int"),
        F.col("driver_key").cast("int"),
        F.col("route_key").cast("int"),
        F.col("shipment_key").cast("int"),
        F.col("date_key").cast("int"),
        get_field(incremental_telemetry_df, ["trip_id"], "string").alias("trip_id"),
        get_field(incremental_telemetry_df, ["latitude"], "decimal(9,6)").alias("latitude"),
        get_field(incremental_telemetry_df, ["longitude"], "decimal(9,6)").alias("longitude"),
        get_field(incremental_telemetry_df, ["speed_kph", "speed_kmh"], "decimal(6,2)").alias("speed_kmh"),
        get_field(incremental_telemetry_df, ["fuel_rate_lph", "fuel_consumption_lph"], "decimal(6,2)").alias("fuel_consumption_lph"),
        F.lit(None).cast("decimal(8,4)").alias("fuel_used_liters_delta"),
        get_field(incremental_telemetry_df, ["rpm"], "int").alias("rpm"),
        get_field(incremental_telemetry_df, ["engine_temperature_c", "engine_temp_c"], "decimal(5,2)").alias("engine_temp_c"),
        get_field(incremental_telemetry_df, ["cargo_temperature_c", "cargo_temp_c"], "decimal(5,2)").alias("cargo_temp_c"),
        get_field(incremental_telemetry_df, ["ignition_on"], "boolean").alias("ignition_on"),
        get_field(incremental_telemetry_df, ["odometer_km"], "decimal(12,2)").alias("odometer_km"),
        F.lit("telemetry_event").alias("event_type")
    ).dropDuplicates(["event_id"])

write_to_sql(fct_telemetry_df, "dw.fct_vehicle_telemetry", mode="append")

print("=== ALL 5 DIMENSIONS AND 3 FACTS PUBLISHED TO AZURE SQL ===")

dw.dim_vehicle already populated. Skipping...
dw.dim_driver already populated. Skipping...
dw.dim_route already populated. Skipping...
dw.dim_shipment already populated. Skipping...
dw.fct_trip already populated. Skipping...
No new records to write to dw.fct_cold_chain. Skipping...
Current Watermark in dw.fct_vehicle_telemetry: 2026-10-02 09:19:00
Writing 166000 rows to dw.fct_vehicle_telemetry (mode: append)...
Successfully published dw.fct_vehicle_telemetry
=== ALL 5 DIMENSIONS AND 3 FACTS PUBLISHED TO AZURE SQL ===
